#  TD(λ) WITH SARSA Agent

Prompt:
Build a SARSA(λ) agent using TD(λ) for OpenAI Gymnasium's Taxi-v3 environment. 
The agent should:
 - Implement SARSA with eligibility traces.
 - Use a Q-table to store state-action values.
 - Train over multiple episodes with ε-greedy exploration.
 - Debug eligibility traces, TD error, and training performance.
 - Evaluate the agent’s performance over multiple test episodes.
 - Display the average reward and visualize the trained agent in action.
 - Extract passenger and destination locations from state encoding for better understanding.
 - Add debugging steps to track learning progress.

OpenAI. (2025). SARSA(λ) with TD(λ) for Taxi-v3 using OpenAI Gymnasium: A Reinforcement Learning Approach [ChatGPT response]. OpenAI. Retrieved March 21, 2025, from https://chat.openai.com

In [6]:
import gymnasium as gym
# Create the environment with a rendering mode that works in Jupyter
env = gym.make("Taxi-v3", render_mode="ansi")

In [12]:
# Crear el entorno Taxi-v3
env = gym.make("Taxi-v3", render_mode="ansi")

# Ver el número de estados y acciones posibles
print("Espacio de estados:", env.observation_space)
print("Espacio de acciones:", env.action_space)

Espacio de estados: Discrete(500)
Espacio de acciones: Discrete(6)


In [9]:
initial_state, info = env.reset()

print("initial state:", initial_state)

# Tomar una acción (por ejemplo, moverse hacia el sur = 0)
new_obs, reward, finish, truncated, info = env.step(0)

print("New observation:", new_obs)
print("reward:", reward)
print("¿Episode finished?:", finish)
print("¿truncated?:", truncated)
print("Info:", info)


initial state: 44
New observation: 144
reward: -1
¿Episode finished?: False
¿truncated?: False
Info: {'prob': 1.0, 'action_mask': array([1, 1, 1, 0, 0, 0], dtype=int8)}


# Visualize the environment on a random state

In [21]:
# Reset the environment
state, info = env.reset()

# Render the initial state
print(env.render())


+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




# Run a random episode

In [46]:
import random
import time

# Reset the environment
state, info = env.reset()

# Run a random episode
for _ in range(10):  # Limit to 10 steps for now
    action = env.action_space.sample()  # Pick a random action
    new_state, reward, done, truncated, info = env.step(action)  # Take the action
    
    # Render the environment after the step
    print(env.render())

    # Add a small delay to visualize better
    time.sleep(0.5)

    # Stop if the episode is finished
    if done:
        break


+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (East)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Pickup)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (West)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Dropoff)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (East)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (North)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (South)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (East)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Dropoff)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Pickup)



# Set Up the SARSA(λ) Agent

Create the Q-table: This stores the expected reward for each (state, action) pair.

Create the Eligibility Trace table: This tracks how much past state-action pairs should be updated.

* Define hyperparameters:
* alpha (α): Learning rate
* gamma (γ): Discount factor
* lambda (λ): Controls how far back we remember
* epsilon (ε): Exploration vs. exploitation balance

In [23]:
import numpy as np

# Initialize Q-table with zeros (500 states x 6 actions)
Q_table = np.zeros((env.observation_space.n, env.action_space.n))

# Initialize eligibility trace table with zeros (same shape as Q-table)
E_table = np.zeros((env.observation_space.n, env.action_space.n))

# Set hyperparameters
alpha = 0.1  # Learning rate
gamma = 0.99  # Discount factor (future rewards importance)
lmbda = 0.9  # Lambda for eligibility traces (closer to 1 = longer memory)
epsilon = 0.1  # Exploration rate (probability of taking random actions)


# Implement SARSA(λ) Training

In [36]:
# Training parameters
num_episodes = 10000  # Total episodes to train the agent
max_steps_per_episode = 100  # Max steps per episode to avoid infinite loops

for episode in range(num_episodes):
    # Reset environment
    state, info = env.reset()
    
    # Choose initial action using epsilon-greedy
    if np.random.rand() < epsilon:
        action = env.action_space.sample()  # Explore
    else:
        action = np.argmax(Q_table[state, :])  # Exploit (best known action)
    
    # Reset eligibility traces
    E_table.fill(0)

    for step in range(max_steps_per_episode):
        # Take action, observe reward and new state
        new_state, reward, done, truncated, info = env.step(action)

        # Choose next action using epsilon-greedy
        if np.random.rand() < epsilon:
            new_action = env.action_space.sample()  # Explore
        else:
            new_action = np.argmax(Q_table[new_state, :])  # Exploit

        # Compute TD error
        td_error = reward + gamma * Q_table[new_state, new_action] - Q_table[state, action]

        # Update eligibility trace for (state, action)
        E_table[state, action] += 1

        # Update Q-table and decay eligibility traces
        Q_table += alpha * td_error * E_table
        E_table *= gamma * lmbda  # Decay traces

        # Debugging: Print values every 1000 episodes & first few steps
        if episode % 1000 == 0 and step < 3:  # Print only first 3 steps per episode to avoid too much output
            print(f"Episode {episode}, Step {step}:")
            print(f"  State: {state}, Action: {action}, Reward: {reward}")
            print(f"  New State: {new_state}, New Action: {new_action}")
            print(f"  TD Error: {td_error}")
            print(f"  Eligibility Trace (State-Action): {E_table[state, action]}")
            print("-" * 40)

        # Move to the next state-action pair
        state = new_state
        action = new_action

        # Stop if episode is done
        if done:
            break

    # Print progress every 1000 episodes
    if (episode + 1) % 1000 == 0:
        print(f"Episode {episode + 1}/{num_episodes} completed.")

print("Training finished!")


Episode 0, Step 0:
  State: 382, Action: 3, Reward: -1
  New State: 362, New Action: 1
  TD Error: -0.02912862509396641
  Eligibility Trace (State-Action): 0.891
----------------------------------------
Episode 0, Step 1:
  State: 362, Action: 1, Reward: -1
  New State: 262, New Action: 3
  TD Error: 0.8176907522974501
  Eligibility Trace (State-Action): 0.891
----------------------------------------
Episode 0, Step 2:
  State: 262, Action: 3, Reward: -1
  New State: 242, New Action: 3
  TD Error: 1.5845428065233302
  Eligibility Trace (State-Action): 0.891
----------------------------------------
Episode 1000/10000 completed.
Episode 1000, Step 0:
  State: 392, Action: 0, Reward: -1
  New State: 492, New Action: 4
  TD Error: -22.377136576941638
  Eligibility Trace (State-Action): 0.891
----------------------------------------
Episode 1000, Step 1:
  State: 492, Action: 4, Reward: -10
  New State: 492, New Action: 3
  TD Error: 14.075285189096018
  Eligibility Trace (State-Action): 0.

# Evaluate the Trained Agent

In [38]:
num_test_episodes = 100  # Number of test episodes
max_steps_per_episode = 100  # Prevent infinite loops
total_rewards = []  # Store rewards per episode

print("Starting evaluation...")

for episode in range(num_test_episodes):
    state, info = env.reset()
    total_reward = 0  # Track reward for this episode
    done = False

    for step in range(max_steps_per_episode):  # Add a step limit
        action = np.argmax(Q_table[state, :])  # Always pick the best action
        new_state, reward, done, truncated, info = env.step(action)
        total_reward += reward  # Accumulate reward
        state = new_state  # Move to the next state

        if done:  # Stop when the episode ends
            break

    total_rewards.append(total_reward)  # Store reward for this episode

    # Debugging: Print progress every 10 episodes
    if (episode + 1) % 10 == 0:
        print(f"Episode {episode + 1}/{num_test_episodes} - Reward: {total_reward}, Steps Taken: {step + 1}")

# Compute average reward
average_reward = np.mean(total_rewards)
print(f"Evaluation completed! Average reward over {num_test_episodes} episodes: {average_reward}")


Starting evaluation...
Episode 10/100 - Reward: 13, Steps Taken: 8
Episode 20/100 - Reward: 10, Steps Taken: 11
Episode 30/100 - Reward: 6, Steps Taken: 15
Episode 40/100 - Reward: 8, Steps Taken: 13
Episode 50/100 - Reward: 9, Steps Taken: 12
Episode 60/100 - Reward: 4, Steps Taken: 17
Episode 70/100 - Reward: 8, Steps Taken: 13
Episode 80/100 - Reward: 6, Steps Taken: 15
Episode 90/100 - Reward: 3, Steps Taken: 18
Episode 100/100 - Reward: 8, Steps Taken: 13
Evaluation completed! Average reward over 100 episodes: 6.84


# Visualize the Agent in Action

In [45]:
import time

state, info = env.reset()
done = False
locations = ["R", "G", "Y", "B"]


while not done:
    action = np.argmax(Q_table[state, :])  # Best action
    state, reward, done, truncated, info = env.step(action)
    # Extract passenger and destination info
    # Decode state into taxi position, passenger location, and destination
    taxi_row, taxi_col, passenger_loc, destination_loc = env.unwrapped.decode(state)
    
    # Map passenger & destination to actual letters
    passenger_location = locations[passenger_loc] if passenger_loc < 4 else "In Taxi"
    destination = locations[destination_loc]
    
    print(f"Passenger starts at: {passenger_location}, Destination: {destination}")
    # Render the environment
    print(env.render())
    time.sleep(0.5)  # Slow down to observe

print("Final episode complete!")


Passenger starts at: G, Destination: Y
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (West)

Passenger starts at: G, Destination: Y
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (North)

Passenger starts at: G, Destination: Y
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (North)

Passenger starts at: G, Destination: Y
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (North)

Passenger starts at: G, Destination: Y
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (North)

Passenger starts at: G, Destination: Y
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (East)

Passenger starts at: In Taxi, Destination: Y
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Pickup)

Passenger starts at: In Taxi, Destination: Y
+---------+


- We can compare how the agent was correctly trained and we can compare the random episode vs the evaluation one, the taxi is taking a very good route to achieve the goal.
- We also noticed the reward is showing only positive rewards, which is amazing!